## PX1 / Mini-Cam — локальная проверка механизма обновлений

Проверяем **собственный SD-образ** `ЛП.imgc`, не подключаясь к серверам производителя и не запуская ни одного файла из прошивки. В отчёт попадут только названия утилит и файлов, типы/размеры и **протокол с доменом** возможных URL — без логинов, паролей, путей, параметров ссылок или содержимого скриптов. ZIP изучается как архив, без распаковки файлов. По итогу будет создан приватный `PX1_SD_UPDATE_OFFLINE_REPORT.txt` на Google Диске.

Открой Colab на Android и нажми **«Среда выполнения» → «Выполнить все»**. Для временной копии Linux-раздела требуется примерно 9 ГБ на Colab; исходный образ не изменяется.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, struct, shutil
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
src = root / 'Untitled folder' / 'ЛП.imgc'
if not src.is_file():
    src = next((Path(d)/'ЛП.imgc' for d,_,names in os.walk(root) if 'ЛП.imgc' in names), None)
if not src or not src.is_file(): raise FileNotFoundError('Не найден ЛП.imgc')
project=Path('/content/unimgc')
if not project.is_dir():
    subprocess.run(['git','clone','--depth','1','https://github.com/Shizmob/unimgc.git',str(project)],check=True)
subprocess.run(['make','-C',str(project)],check=True)
part=Path('/content/px1_ro_partition.img')
offset=32901120
size=7912719360
if shutil.disk_usage('/content').free < size + 1024**3:
    raise RuntimeError('Мало свободного места в Colab для копии раздела')
print('Читаю раздел в временный файл Colab (примерно 7.9 ГБ)...')
p=subprocess.Popen([str(project/'unimgc'),str(src)],stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
try:
    toskip=offset
    while toskip:
        data=p.stdout.read(min(4*1024*1024,toskip))
        if not data: raise EOFError('Образ закончился до начала раздела')
        toskip-=len(data)
    left=size
    with part.open('wb') as out:
        while left:
            data=p.stdout.read(min(4*1024*1024,left))
            if not data: raise EOFError('Образ закончился внутри раздела')
            out.write(data)
            left-=len(data)
finally:
    p.stdout.close()
    p.terminate()
    try: p.wait(timeout=10)
    except subprocess.TimeoutExpired: p.kill(); p.wait()
def dbg(command):
    v=subprocess.run(['debugfs','-R',command,str(part)],capture_output=True,timeout=90)
    return v.stdout.decode('utf-8','replace')

# PASSIVE INSPECTION ONLY. This notebook makes no network requests to addresses found in the firmware.
import re, zipfile, ipaddress, urllib.parse, hashlib
report=[
  "PX1 Mini-Cam — offline audit of firmware update artifacts",
  "Source: owner-controlled SD image; strings, credentials and full URLs are not exported.",
]
def add(k,v):report.append(f"{k}: {v}")
def fs_stat(path):
    raw=dbg('stat '+path)
    a=re.search(r'Type:\s*(\w+)',raw)
    n=re.search(r'Size:\s*(\d+)',raw)
    return (a.group(1) if a else None, int(n.group(1)) if n else -1)
def list_names(path, limit=60):
    raw=dbg('ls -p '+path)
    names=[]
    for row in raw.splitlines():
        fields=row.strip().split('/')
        if len(fields)>=6 and fields[5] and fields[5] not in ('.','..'):
            names.append(fields[5])
    return names[:limit]
def pull(path,out):
    out.unlink(missing_ok=True)
    # Filename/path is a fixed, internal absolute path, never user-supplied.
    dbg('dump '+path+' '+str(out))
    return out.is_file()
scripts=[
    '/root/GetSoftwareUpdate.sh','/root/startup.sh','/root/RunOnceScript.sh',
    '/root/startSamba.sh','/root/wifi_boot.conf',
    '/etc/rc.d/rc.local','/etc/rc.d/rcS',
]
tokens=[
  'curl','wget','ftp','tftp','scp','sftp','ssh','dropbear','boa',
  'httpd','busybox','unzip','tar','gzip','gunzip','md5sum','sha1sum',
  'sha256sum','openssl','mount','umount','dd','fw_setenv','fw_printenv',
  'mtd','flash_erase','nandwrite','ubiupdatevol','reboot','ProteusApp',
  'UDProteus','wpa_supplicant'
]
def url_host_metadata(raw):
    # Show hostnames only, never username/password/path/query/fragment or embedded token.
    result=set()
    for u in re.findall(r'(?:https?|ftps?|tftp)://[^\s\'"<>\\]+',raw,re.I):
        try:
            parsed=urllib.parse.urlsplit(u)
            host=(parsed.hostname or '').lower().rstrip('.')
            if not host:continue
            if len(host)>180:continue
            try:
                ip=ipaddress.ip_address(host)
                public=ip.is_global
            except ValueError: public=not(host.endswith(('.local','.internal','.lan','.localhost')) or '.' not in host)
            if public and re.fullmatch(r'[a-z0-9.-]+',host):
                result.add(parsed.scheme.lower()+'://'+host)
            else:
                result.add(parsed.scheme.lower()+'://[private-or-local-host]')
        except Exception:pass
    return sorted(result)
for path in scripts:
    ftype, size=fs_stat(path)
    add('Script '+path+' file type / bytes',str(ftype)+' / '+str(size))
    if ftype!='regular' or size<0 or size>250000:
        continue
    raw=dbg('cat '+path)
    # No raw code, hardcoded secrets, script arguments or URL paths go into report.
    if len(raw)>250000:continue
    active=[s for s in raw.splitlines() if s.strip() and not s.lstrip().startswith('#')]
    seen=[t for t in tokens if any(re.search(r'(?<![A-Za-z_])'+re.escape(t)+r'(?![A-Za-z_])',line,re.I) for line in active)]
    add('Script '+path+' active lines',len(active))
    add('Script '+path+' tool names',', '.join(seen) if seen else 'none recognized')
    urls=url_host_metadata('\n'.join(active))
    add('Script '+path+' URL scheme and hostname only',', '.join(urls) if urls else 'not found')
    add('Script '+path+' contains update/transfer hint',str(any(x in raw.lower() for x in ['update','firmware','download','wget','curl','ftp','scp'])))
add('Folder /root/Firmware children',', '.join(list_names('/root/Firmware',60)) or 'empty/unavailable')
for name in ['~~factory','~~V6-4-6-G13','~~PROTEUS_V4.70']:
    path='/root/Firmware/'+name
    ftype,size=fs_stat(path)
    add('Firmware entry '+name+' type/bytes',str(ftype)+' / '+str(size))
    if ftype=='directory':
        add('Firmware entry '+name+' files (up to 40)',', '.join(list_names(path,40)) or 'empty/unavailable')
archive='/root/factoryupdate.zip'
ftype,size=fs_stat(archive)
add('Archive '+archive+' type/bytes',str(ftype)+' / '+str(size))
temp=Path('/content/px1_factoryupdate_inventory.zip')
if ftype=='regular' and 0<size<=250*1024*1024 and pull(archive,temp):
    try:
        with zipfile.ZipFile(temp,'r') as z:
            items=z.infolist()
            add('Archive ZIP file count',len(items))
            add('Archive combined uncompressed size',sum(i.file_size for i in items))
            # Names only; no files are extracted from ZIP.
            for info in items[:80]:
                s=info.filename
                if s.startswith('/') or '..' in s.split('/'):
                    add('Archive suspicious entry name','redacted')
                elif re.fullmatch(r'[A-Za-z0-9_./+@ -]{1,150}',s):
                    add('Archive entry',s+' ('+str(info.file_size)+' bytes)')
                else: add('Archive entry','[nonstandard-name redacted] ('+str(info.file_size)+' bytes)')
    except (zipfile.BadZipFile, OSError, RuntimeError) as e:
        add('Archive ZIP metadata status',type(e).__name__)
    finally:temp.unlink(missing_ok=True)
else:
    add('Archive ZIP inspection','not attempted (missing, large, or extraction failed)')
report.append('Nothing was uploaded outside the owner Drive, no update URLs were contacted.')
out=root/'PX1_SD_UPDATE_OFFLINE_REPORT.txt'
out.write_text('\n'.join(report)+'\n',encoding='utf-8')
part.unlink(missing_ok=True)
print('\n'.join(report))
print('Готово:',out)
